# 06 · Silver — physical_itens_venda_caixa
**Projeto:** Estágio Engenharia de Dados · **Squad:** 3 — Batch · **Autor:** Alexandro Silva

Segunda camada da arquitetura Medallion. Lê a Bronze, aplica as regras técnicas da Silver,
adiciona `silver_processed_at` e grava em Delta particionado por `ano` / `mes`.

## Regras geradas por este notebook
Regras técnicas de `physical_itens_venda_caixa` (planilha de regras da Squad 3):

| Nº | Regra | Tratamento |
|---|---|---|
| 1 | `id_item_venda` não pode ser nulo nem duplicado | linha removida (fica a ingestão mais recente) |
| 2 | `id_transacao` deve existir em `physical_vendas_caixa` (FK) | item órfão removido |
| 3 | `quantidade` > 0; kg/L aceita decimal, `un` deve ser inteiro | ≤ 0 removida; `un` fracionado fica com `flag_quantidade_fracionada_un` |
| 4 | `preco_unitario_registro` > 0 e `DECIMAL(10,2)` | cast aplicado; ≤ 0 removida |
| 5 | `valor_total_item` = `preco_unitario_registro` × `quantidade` | divergência acima do arredondamento (R$ 0,01 + 0,05% do preço) fica com `flag_valor_inconsistente` |

**KPIs:** nenhum. As regras 6 a 10 da planilha são KPIs e enriquecimentos da camada Gold.

## Colunas adicionadas
| Coluna | Conteúdo |
|---|---|
| `unidade_medida` | unidade do produto (kg, l, un), vinda do catálogo de produtos |
| `flag_quantidade_fracionada_un` | produto em `un` com quantidade não inteira (regra 3) |
| `flag_valor_inconsistente` | valor total fora da tolerância de arredondamento (regra 5) |
| `silver_processed_at` | data e hora do processamento na Silver |

## Origem e destino
| Item | Definição |
|---|---|
| Origem | `abfss://squad3@internshipdatalake.dfs.core.windows.net/bronze/physical_itens_venda_caixa` |
| Destino | `abfss://squad3@internshipdatalake.dfs.core.windows.net/silver/physical_itens_venda_caixa` |
| Formato | Delta, particionado por `ano` / `mes` |
| Modo | upsert pela chave `id_item_venda` (equivalente ao `MERGE`; ver etapa 6) |

## Etapas
1. Credenciais e caminhos
2. Leitura da Bronze
3. Tipagem, padronização e regra 1
4. Avaliação das regras 2 a 5
5. Aplicação das regras e coluna de auditoria
6. Gravação na Silver
7. Validação

## 1. Credenciais e caminhos
Carrega as credenciais do `.env` e define os caminhos da Bronze e da Silver no container `squad3`.
As tabelas de referência das regras 2 e 3 vêm **provisoriamente** das cópias derivadas da RAW; devem
passar a vir da Silver das duplas responsáveis assim que forem publicadas.

In [0]:
# Célula 1 — Credenciais e caminhos
from dotenv import load_dotenv
import os

load_dotenv("/Workspace/Users/alexandrofs31@gmail.com/.env")

CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
TENANT_ID       = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT", "internshipdatalake")

faltando = [n for n, v in {"ADLS_CLIENT_ID": CLIENT_ID, "ADLS_TENANT_ID": TENANT_ID,
                           "ADLS_CLIENT_SECRET": CLIENT_SECRET}.items() if not v]
if faltando:
    raise ValueError(f"Variáveis ausentes no .env: {faltando}")

TABELA          = "physical_itens_venda_caixa"
CONTAINER_SQUAD = "squad3"   # camadas ficam DENTRO do container da squad
BASE_SQUAD      = f"abfss://{CONTAINER_SQUAD}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
CAMINHO_BRONZE  = f"{BASE_SQUAD}/bronze/{TABELA}"
CAMINHO_SILVER  = f"{BASE_SQUAD}/silver/{TABELA}"

# True = lê a Bronze do container squad3 (notebook 05 atual).
# False = lê a Bronze antiga, do Unity Catalog.
LER_BRONZE_DO_ADLS = True
TABELA_BRONZE_UC   = f"workspace.squad3.bronze_{TABELA}"

# Tabelas de referência das regras 2 e 3.
# PROVISÓRIO: são as cópias derivadas da RAW. O professor exige que venham da SILVER
# das duplas responsáveis; troque assim que elas publicarem.
TABELA_VENDAS_REF      = "workspace.squad3.physical_vendas_caixa"
TABELA_PERECIVEIS_REF  = "workspace.squad3.physical_produtos_pereciveis"
TABELA_PRODUTOS_REF    = "workspace.squad3.ecommerce_produtos"

sufixo = f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
opcoes_adls = {
    f"fs.azure.account.auth.type.{sufixo}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{sufixo}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{sufixo}": CLIENT_ID,
    f"fs.azure.account.oauth2.client.secret.{sufixo}": CLIENT_SECRET,
    f"fs.azure.account.oauth2.client.endpoint.{sufixo}": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

print(f"📂 Bronze : {CAMINHO_BRONZE if LER_BRONZE_DO_ADLS else TABELA_BRONZE_UC}")
print(f"📦 Silver : {CAMINHO_SILVER}")

## 2. Leitura da Bronze
Lê a Bronze em Delta. Como ela é `append`, pode conter mais de uma carga do mesmo arquivo; a
deduplicação é feita na etapa 3.

In [0]:
# Célula 2 — Leitura da Bronze
from pyspark.sql import functions as F
from pyspark.sql.window import Window

if LER_BRONZE_DO_ADLS:
    df_bronze = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_BRONZE)
else:
    df_bronze = spark.table(TABELA_BRONZE_UC)

total_bronze = df_bronze.count()
print(f"✅ Bronze lida: {total_bronze:,} linhas | {len(df_bronze.columns)} colunas")
df_bronze.printSchema()

## 3. Tipagem, padronização e regra 1
Converte as colunas para os tipos definitivos (valores monetários em `DECIMAL`, e não `double`, para
não haver erro de arredondamento), remove espaços e padroniza o código do produto em minúsculas.
Aplica a **regra 1**: remove `id_item_venda` nulo e, havendo duplicata, mantém a ingestão mais recente.

In [0]:
# Célula 3 — Tipagem, padronização e Regra 1 (id_item_venda não nulo e único)

df_tipado = df_bronze.select(
    F.col("id_item_venda").cast("long").alias("id_item_venda"),
    F.trim(F.col("id_transacao")).alias("id_transacao"),
    F.lower(F.trim(F.col("codigo_barras_produto"))).alias("codigo_barras_produto"),
    F.col("quantidade").cast("decimal(12,3)").alias("quantidade"),
    F.col("preco_unitario_registro").cast("decimal(10,2)").alias("preco_unitario_registro"),  # Regra 4
    F.col("valor_total_item").cast("decimal(12,2)").alias("valor_total_item"),
    F.col("ano").cast("int").alias("ano"),
    F.col("mes").cast("int").alias("mes"),
    F.col("bronze_ingested_at"),
    F.col("bronze_source_file"),
)

# Regra 1a — PK não nula
df_com_chave = df_tipado.filter(F.col("id_item_venda").isNotNull())
pk_nula = total_bronze - df_com_chave.count()

# Regra 1b — PK única. Como a Bronze é append, o mesmo item pode vir de várias
# ingestões: fica a versão mais recente de cada id_item_venda
janela = Window.partitionBy("id_item_venda").orderBy(F.col("bronze_ingested_at").desc())
df_dedup = (df_com_chave
    .withColumn("_rn", F.row_number().over(janela))
    .filter(F.col("_rn") == 1)
    .drop("_rn"))

total_dedup = df_dedup.count()
pk_duplicada = total_bronze - pk_nula - total_dedup
print(f"Regra 1 | id_item_venda nulo      : {pk_nula:,} removidas")
print(f"Regra 1 | id_item_venda duplicado : {pk_duplicada:,} removidas")
print(f"✅ Linhas após a Regra 1          : {total_dedup:,}")

## 4. Avaliação das regras 2 a 5
Marca cada linha quanto às regras 2 a 5 e imprime a contagem de violações, **sem remover nada ainda**.
- **Regra 2:** cruza com `physical_vendas_caixa` para achar itens sem venda pai.
- **Regra 3:** busca a unidade de medida no catálogo unificado (`ecommerce_produtos` + `physical_produtos_pereciveis`).
- **Regra 5:** a quantidade vem com 3 casas decimais, então preço × quantidade pode diferir do valor
  cobrado em até 0,0005 × preço. A tolerância é R$ 0,01 + 0,05% do preço; só o que passa disso é marcado.

Qualquer contagem diferente de zero deve ser informada ao professor antes da entrega.

In [0]:
# Célula 4 — Regras 2 a 5: avaliação linha a linha

# Regra 2 — vendas existentes (FK)
df_vendas_ref = (spark.table(TABELA_VENDAS_REF)
    .select(F.trim(F.col("id_transacao")).alias("id_transacao"))
    .distinct()
    .withColumn("_venda_existe", F.lit(True)))

# Regra 3 — unidade de medida vem do catálogo unificado (secos + perecíveis)
df_catalogo = (spark.table(TABELA_PRODUTOS_REF).select("sku", "unidade_medida")
    .unionByName(spark.table(TABELA_PERECIVEIS_REF).select("sku", "unidade_medida"))
    .select(F.lower(F.trim(F.col("sku"))).alias("codigo_barras_produto"),
            F.lower(F.trim(F.col("unidade_medida"))).alias("unidade_medida"))
    .dropDuplicates(["codigo_barras_produto"]))

# Regra 5 — tolerância de arredondamento. A quantidade vem com 3 casas decimais, então
# preço × quantidade pode diferir do valor cobrado em até 0,0005 × preço (R$ 0,15 no
# maior preço da base), mais R$ 0,01 do arredondamento do próprio valor.
TOLERANCIA_FIXA     = 0.01
TOLERANCIA_POR_REAL = 0.0005

df_avaliado = (df_dedup
    .join(df_vendas_ref, on="id_transacao", how="left")
    .join(F.broadcast(df_catalogo), on="codigo_barras_produto", how="left")
    .withColumn("_r2_fk_ok",  F.col("_venda_existe").isNotNull())
    .withColumn("_r3_qtd_ok", F.coalesce(F.col("quantidade") > 0, F.lit(False)))
    .withColumn("_r4_preco_ok", F.coalesce(F.col("preco_unitario_registro") > 0, F.lit(False)))
    # Regra 3 — produto vendido em 'un' deveria ter quantidade inteira
    .withColumn("flag_quantidade_fracionada_un",
        F.coalesce((F.col("unidade_medida") == "un") &
                   (F.col("quantidade") != F.floor(F.col("quantidade"))), F.lit(False)))
    # Regra 5 — valor_total_item deve bater com preço × quantidade
    .withColumn("flag_valor_inconsistente",
        F.coalesce(F.abs(F.round(F.col("quantidade") * F.col("preco_unitario_registro"), 2)
                         - F.col("valor_total_item"))
                   > (F.lit(TOLERANCIA_FIXA) + F.col("preco_unitario_registro") * F.lit(TOLERANCIA_POR_REAL)),
                   F.lit(True)))
)

def conta(cond):
    return F.sum(cond.cast("int"))

r = df_avaliado.agg(
    conta(~F.col("_r2_fk_ok")).alias("r2_sem_venda_pai"),
    conta(~F.col("_r3_qtd_ok")).alias("r3_quantidade_invalida"),
    conta(F.col("unidade_medida").isNull()).alias("r3_produto_fora_do_catalogo"),
    conta(F.col("flag_quantidade_fracionada_un")).alias("r3_un_com_quantidade_fracionada"),
    conta(~F.col("_r4_preco_ok")).alias("r4_preco_invalido"),
    conta(F.col("flag_valor_inconsistente")).alias("r5_valor_inconsistente"),
).collect()[0].asDict()

acao = {
    "r2_sem_venda_pai": "removida",
    "r3_quantidade_invalida": "removida",
    "r3_produto_fora_do_catalogo": "mantida (só reporta)",
    "r3_un_com_quantidade_fracionada": "mantida com flag",
    "r4_preco_invalido": "removida",
    "r5_valor_inconsistente": "mantida com flag",
}

print("=" * 78)
print("VALIDAÇÃO DAS REGRAS — physical_itens_venda_caixa")
print("=" * 78)
for regra, qtd in r.items():
    qtd = qtd or 0
    marca = "✅" if qtd == 0 else "⚠️ "
    print(f"{marca} {regra:<34} {qtd:>10,} ({qtd / total_dedup * 100:6.2f}%)  {acao[regra]}")
print("\nTudo que estiver com ⚠️ deve ser informado ao professor antes da entrega.")

## 5. Aplicação das regras e coluna de auditoria
Remove as linhas que violam as regras 2, 3 (quantidade ≤ 0) e 4. As regras 3 (`un` fracionado) e 5
não removem: a linha fica na Silver com a flag correspondente, para não distorcer a receita na Gold.
Adiciona `silver_processed_at`.

In [0]:
# Célula 5 — Aplica as regras e adiciona a coluna de auditoria

df_silver = (df_avaliado
    .filter(F.col("_r2_fk_ok") & F.col("_r3_qtd_ok") & F.col("_r4_preco_ok"))
    .select(
        "id_item_venda", "id_transacao", "codigo_barras_produto", "unidade_medida",
        "quantidade", "preco_unitario_registro", "valor_total_item",
        "flag_quantidade_fracionada_un", "flag_valor_inconsistente",
        "ano", "mes", "bronze_ingested_at", "bronze_source_file",
    )
    .withColumn("silver_processed_at", F.current_timestamp()))

total_silver_df = df_silver.count()
print(f"Linhas na entrada (após Regra 1) : {total_dedup:,}")
print(f"Linhas removidas pelas regras    : {total_dedup - total_silver_df:,}")
print(f"✅ Linhas que vão para a Silver   : {total_silver_df:,}")
print("\nColunas finais:")
for col_ in df_silver.columns:
    print(f"   • {col_}")

## 6. Gravação na Silver
Grava em Delta particionado por `ano` / `mes`, com upsert pela chave `id_item_venda`: chave existente
é atualizada, chave nova é inserida e a que não veio na carga é mantida.

O comando `MERGE` do Delta não funciona neste ambiente: com credenciais por operação retorna
`AZURE_INVALID_CREDENTIALS_CONFIGURATION`, e o `spark.conf.set` das chaves `fs.azure` é bloqueado no
Serverless (`CONFIG_NOT_AVAILABLE`). Por isso o upsert é feito com leitura e gravação, que têm o
mesmo resultado.

In [0]:
# Célula 6 — Gravação Delta na Silver (upsert por id_item_venda), particionada por ano / mes
#
# O comando MERGE do Delta não aceita as credenciais OAuth por operação no Databricks
# Serverless (erro AZURE_INVALID_CREDENTIALS_CONFIGURATION). O mesmo resultado é obtido
# com leitura + gravação, que aceitam: linhas com chave já existente são atualizadas,
# chaves novas são inseridas e as que não vieram nesta carga são mantidas.

# True = recria a Silver do zero (use UMA vez, quando o schema mudar). Depois volte para False.
RECRIAR_SILVER = False

try:
    df_atual = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_SILVER)
    df_atual.limit(1).collect()
    silver_existe = True
except Exception:
    silver_existe = False

if RECRIAR_SILVER or not silver_existe:
    df_final = df_silver
    print("Primeira carga ou recriação: a Silver será gravada por completo.")
else:
    chaves_novas = df_silver.select("id_item_venda")
    df_mantidas  = df_atual.join(chaves_novas, on="id_item_venda", how="left_anti")
    inseridas    = chaves_novas.join(df_atual.select("id_item_venda"), on="id_item_venda", how="left_anti").count()
    mantidas     = df_mantidas.count()
    print(f"Upsert | atualizadas : {total_silver_df - inseridas:,}")
    print(f"Upsert | inseridas   : {inseridas:,}")
    print(f"Upsert | mantidas    : {mantidas:,}")
    df_final = df_silver.unionByName(df_mantidas)

(df_final.write
    .format("delta")
    .options(**opcoes_adls)
    .mode("overwrite")
    .option("overwriteSchema", "true" if RECRIAR_SILVER else "false")
    .partitionBy("ano", "mes")
    .save(CAMINHO_SILVER))

print("✅ Silver gravada.")
print(f"   Destino   : {CAMINHO_SILVER}")
print("   Partições : ano / mes")

## 7. Validação
Relê a Silver gravada e confere o total de linhas, a unicidade de `id_item_venda` (a execução é
interrompida se houver chave duplicada), as linhas por partição, uma amostra e o schema final.

In [0]:
# Célula 7 — Validação da Silver gravada

df_val = spark.read.format("delta").options(**opcoes_adls).load(CAMINHO_SILVER)

total_silver = df_val.count()
chaves_unicas = df_val.select("id_item_venda").distinct().count()

print(f"✅ Linhas na Silver        : {total_silver:,}")
print(f"✅ id_item_venda distintos : {chaves_unicas:,}")
if total_silver != chaves_unicas:
    raise ValueError("Chave duplicada na Silver: o upsert não deveria permitir isso.")

print("\n📅 Linhas por partição:")
df_val.groupBy("ano", "mes").count().orderBy("ano", "mes").show(40, truncate=False)

print("📋 Amostra:")
df_val.show(3, truncate=False)
df_val.printSchema()